# Assignment 1 – Python/NumPy Basics
**Objective:** Dataset characteristics, data visualization, data analysis  
**Dataset:** Date Fruit Dataset (Kaggle)

## Task 1 – Read the XLSX file using a Pandas DataFrame

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import matplotlib.pyplot as plt

# Read the xlsx file – make sure 'Date_Fruit_Datasets.xlsx' is in the same folder
df_full = pd.read_excel('Date_Fruit_Datasets.xlsx')

print("Full dataset shape:", df_full.shape)
print("Columns:", list(df_full.columns))
df_full.head()

## Task 2 – Keep first, second, third dimensions + class label column

In [ ]:
# Identify the feature columns and the class label column
# The last column is the class label; the first three columns are the first three dimensions
feature_cols = list(df_full.columns[:3])   # first three real-valued dimensions
label_col    = df_full.columns[-1]          # last column = class label

# Keep only four columns: 3 feature dimensions + class label
df = df_full[feature_cols + [label_col]].copy()

print("Reduced dataset shape:", df.shape)
print("Columns kept:", list(df.columns))
df.head()

## Task 3 – Convert DataFrame to NumPy array; encode class labels numerically

In [ ]:
# Store the unique string class names in a list so we can map index -> name later
class_names = list(df[label_col].unique())   # e.g. ['BERHI', 'DEGLET', 'DOKOL', ...]
print("Class names:", class_names)

# Build a mapping from string name to integer index (0, 1, 2, ...)
class_to_idx = {name: idx for idx, name in enumerate(class_names)}
print("Mapping:", class_to_idx)

# Replace string labels with integer indices in the dataframe copy
df['class_idx'] = df[label_col].map(class_to_idx)

# Convert the three feature columns + numeric class index to a NumPy array
# Columns: [dim1, dim2, dim3, class_idx]
data = df[feature_cols + ['class_idx']].to_numpy(dtype=float)

print("NumPy array shape:", data.shape)
print("First 5 rows:\n", data[:5])

## Task 4 – Helper: define per-class Plotly marker symbols

In [ ]:
# We assign a distinct shape to each class for all subsequent plots.
# Plotly supports: 'circle', 'square', 'diamond', 'triangle-up',
#                  'cross', 'x', 'star', 'pentagon', 'hexagram', ...

SHAPES_2D = ['circle', 'square', 'diamond', 'triangle-up',
             'cross', 'x', 'star', 'pentagon', 'hexagram']

SHAPES_3D = ['circle', 'square', 'diamond', 'cross',
             'x', 'circle-open', 'square-open', 'diamond-open', 'circle-dot']

# Assign a color per class (used consistently throughout)
COLORS = px.colors.qualitative.Plotly   # up to 10 distinct colors

num_classes = len(class_names)
print(f"Number of classes: {num_classes}")
for i, name in enumerate(class_names):
    print(f"  Class {i}: {name}  shape2d={SHAPES_2D[i % len(SHAPES_2D)]}  shape3d={SHAPES_3D[i % len(SHAPES_3D)]}")

## Task 5 – 2D Interactive Scatter Plot (first two dimensions, different shapes per class)

In [ ]:
# dim1 = data[:,0], dim2 = data[:,1], class_idx = data[:,3]
dim1 = data[:, 0]
dim2 = data[:, 1]
labels_idx = data[:, 3].astype(int)

fig_2d = go.Figure()

for i, name in enumerate(class_names):
    # Boolean mask: select only the rows that belong to this class
    mask = (labels_idx == i)

    fig_2d.add_trace(go.Scatter(
        x=dim1[mask],
        y=dim2[mask],
        mode='markers',
        name=name,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=8,
            color=COLORS[i % len(COLORS)]
        )
    ))

fig_2d.update_layout(
    title='Task 5 – 2D Scatter Plot (Dim1 vs Dim2)',
    xaxis_title=feature_cols[0],
    yaxis_title=feature_cols[1],
    legend_title='Class'
)

fig_2d.show()

## Task 6 – 3D Interactive Scatter Plot (first three dimensions, different shapes per class)

In [ ]:
dim3 = data[:, 2]

fig_3d = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    fig_3d.add_trace(go.Scatter3d(
        x=dim1[mask],
        y=dim2[mask],
        z=dim3[mask],
        mode='markers',
        name=name,
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=5,
            color=COLORS[i % len(COLORS)]
        )
    ))

fig_3d.update_layout(
    title='Task 6 – 3D Scatter Plot (Dim1, Dim2, Dim3)',
    scene=dict(
        xaxis_title=feature_cols[0],
        yaxis_title=feature_cols[1],
        zaxis_title=feature_cols[2]
    ),
    legend_title='Class'
)

fig_3d.show()

## Task 7 – 2D Scatter + Class Means (repeat Task 5, add larger mean markers)

In [ ]:
fig_2d_mean = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    # --- original data points (same as Task 5, untouched) ---
    fig_2d_mean.add_trace(go.Scatter(
        x=dim1[mask],
        y=dim2[mask],
        mode='markers',
        name=name,
        showlegend=True,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=8,
            color=COLORS[i % len(COLORS)]
        )
    ))

    # --- compute class mean for dim1 and dim2 ---
    # Using numpy mean along axis=0 on the subset rows
    mean_d1 = dim1[mask].mean()   # scalar mean of dimension 1 for this class
    mean_d2 = dim2[mask].mean()   # scalar mean of dimension 2 for this class

    # --- plot the mean with the SAME shape but LARGER size ---
    fig_2d_mean.add_trace(go.Scatter(
        x=[mean_d1],
        y=[mean_d2],
        mode='markers',
        name=f'{name} mean',
        showlegend=True,
        marker=dict(
            symbol=SHAPES_2D[i % len(SHAPES_2D)],
            size=18,                         # larger size to highlight the mean
            color=COLORS[i % len(COLORS)],
            line=dict(width=2, color='black') # black border to make it stand out
        )
    ))

fig_2d_mean.update_layout(
    title='Task 7 – 2D Scatter + Class Means (Dim1 vs Dim2)',
    xaxis_title=feature_cols[0],
    yaxis_title=feature_cols[1],
    legend_title='Class / Mean'
)

fig_2d_mean.show()

## Task 8 – 3D Scatter + Class Means (repeat Task 6, add larger mean markers)

In [ ]:
fig_3d_mean = go.Figure()

for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    # --- original data points (same as Task 6, untouched) ---
    fig_3d_mean.add_trace(go.Scatter3d(
        x=dim1[mask],
        y=dim2[mask],
        z=dim3[mask],
        mode='markers',
        name=name,
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=5,
            color=COLORS[i % len(COLORS)]
        )
    ))

    # --- compute class mean for all three dimensions ---
    mean_d1 = dim1[mask].mean()
    mean_d2 = dim2[mask].mean()
    mean_d3 = dim3[mask].mean()

    # --- plot the 3D mean with the SAME shape but LARGER size ---
    fig_3d_mean.add_trace(go.Scatter3d(
        x=[mean_d1],
        y=[mean_d2],
        z=[mean_d3],
        mode='markers',
        name=f'{name} mean',
        marker=dict(
            symbol=SHAPES_3D[i % len(SHAPES_3D)],
            size=12,                          # larger size for mean
            color=COLORS[i % len(COLORS)],
            line=dict(width=2, color='black')
        )
    ))

fig_3d_mean.update_layout(
    title='Task 8 – 3D Scatter + Class Means',
    scene=dict(
        xaxis_title=feature_cols[0],
        yaxis_title=feature_cols[1],
        zaxis_title=feature_cols[2]
    ),
    legend_title='Class / Mean'
)

fig_3d_mean.show()

## Task 9 – Histograms for each of the first three dimensions (one plot per dimension)

In [ ]:
# Plot one histogram per dimension using matplotlib
for i in range(3):
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.hist(data[:, i], bins=30, color='steelblue', edgecolor='black')
    ax.set_title(f'Task 9 – Histogram of {feature_cols[i]}')
    ax.set_xlabel(feature_cols[i])
    ax.set_ylabel('Frequency')
    plt.tight_layout()
    plt.show()

## Task 10 – Standard Deviation (vectorized, no loops)

In [ ]:
# Extract the first three dimension columns from the numpy array
X = data[:, :3]   # shape: (n_samples, 3)

# --- Step 1: Compute the mean of each dimension ---
# np.sum over axis=0 sums down rows, giving a (3,) vector of column sums
n = X.shape[0]                          # number of samples
mean_vec = np.sum(X, axis=0) / n        # mean for each of the 3 dimensions

# --- Step 2: Centre the data (subtract mean from every row) ---
# Broadcasting: (n,3) - (3,) subtracts the mean vector from each row
X_centered = X - mean_vec              # shape: (n, 3)

# --- Step 3: Compute variance = mean of squared deviations ---
# Element-wise square, then sum down rows, then divide by n
variance_vec = np.sum(X_centered ** 2, axis=0) / n   # shape: (3,)

# --- Step 4: Standard deviation = sqrt of variance ---
std_vec = variance_vec ** 0.5          # shape: (3,)  — no loop, pure vectorized

print("Standard deviations (vectorized, no loop):")
for i in range(3):
    print(f"  {feature_cols[i]}: sigma = {std_vec[i]:.4f}")

## Task 11 – Box Plot for all data points across first three dimensions

In [ ]:
# One box per dimension; x-axis = dimension names, y-axis = values
fig, ax = plt.subplots(figsize=(8, 5))

# Pass the three dimension arrays as a list to boxplot
ax.boxplot([data[:, 0], data[:, 1], data[:, 2]],
           labels=feature_cols[:3],
           patch_artist=True,
           boxprops=dict(facecolor='lightblue'))

ax.set_title('Task 11 – Box Plots for First Three Dimensions (All Data)')
ax.set_xlabel('Dimension')
ax.set_ylabel('Value')
plt.tight_layout()
plt.show()

## Task 12 – Box Plots per Class (one figure per class, three boxes per figure)

In [ ]:
# For each class: plot three boxes (one per dimension) in a single figure
for i, name in enumerate(class_names):
    mask = (labels_idx == i)

    # Subset data for this class
    class_data = data[mask, :3]   # shape: (n_class, 3)

    fig, ax = plt.subplots(figsize=(8, 5))

    ax.boxplot([class_data[:, 0], class_data[:, 1], class_data[:, 2]],
               labels=feature_cols[:3],
               patch_artist=True,
               boxprops=dict(facecolor='lightgreen'))

    ax.set_title(f'Task 12 – Box Plots for Class: {name}')
    ax.set_xlabel('Dimension')
    ax.set_ylabel('Value')
    plt.tight_layout()
    plt.show()

## Task 13 – 3×3 Scatter Matrix (matrix plot) for first three dimensions

In [ ]:
# Build a 3x3 grid of scatter plots
# Diagonal: scatter of a dimension against itself (shows distribution)
# Off-diagonal: scatter of dimension i vs dimension j

fig, axes = plt.subplots(3, 3, figsize=(10, 10))

for row in range(3):
    for col in range(3):
        ax = axes[row][col]

        # Plot each class separately so the classes are colour-coded
        for i, name in enumerate(class_names):
            mask = (labels_idx == i)
            ax.scatter(data[mask, col], data[mask, row],
                       s=5, label=name, alpha=0.6,
                       color=COLORS[i % len(COLORS)])

        # Label axes only on the edges for readability
        if row == 2:
            ax.set_xlabel(feature_cols[col], fontsize=9)
        if col == 0:
            ax.set_ylabel(feature_cols[row], fontsize=9)

# Add a single shared legend outside the grid
handles, labels_leg = axes[0][0].get_legend_handles_labels()
fig.legend(handles, labels_leg, loc='upper right', fontsize=8, title='Class')

fig.suptitle('Task 13 – 3×3 Scatter Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## Task 14 – Circle Segment Plot for first 7 real-valued dimensions

In [ ]:
# -----------------------------------------------------------------------
# Circle Segment (Radviz-style) plot
# Each of the 7 dimensions is placed as an anchor on a unit circle,
# equally spaced at angles 0, 2π/7, 4π/7, ...  A data point is plotted
# at the weighted sum of the anchor positions, where the weights are the
# normalised feature values for that point.
# -----------------------------------------------------------------------

# Use 7 features from the FULL dataset (first 7 real-valued columns)
# We re-read the original dataframe to get 7 dimensions
feature_cols_7 = list(df_full.columns[:7])
X7 = df_full[feature_cols_7].to_numpy(dtype=float)   # shape: (n, 7)

n_dims = 7

# --- Normalise each row so that the row sum is 1 ---
# Shift data so all values are non-negative first
X7_shifted = X7 - X7.min(axis=0)                     # shift min to 0 per column
row_sums   = X7_shifted.sum(axis=1, keepdims=True)    # sum of each row
# Avoid division by zero for any all-zero row
row_sums[row_sums == 0] = 1
X7_norm = X7_shifted / row_sums                       # row-normalised, shape (n, 7)

# --- Anchor positions on the unit circle ---
angles   = np.array([2 * np.pi * k / n_dims for k in range(n_dims)])  # (7,)
anchor_x = np.cos(angles)   # (7,)
anchor_y = np.sin(angles)   # (7,)

# --- Project each data point: point = sum_k( weight_k * anchor_k ) ---
# X7_norm has shape (n, 7); anchor_x has shape (7,)
# Matrix multiply: (n,7) @ (7,) -> (n,)
proj_x = X7_norm @ anchor_x   # shape (n,)
proj_y = X7_norm @ anchor_y   # shape (n,)

# --- Retrieve class indices for all n rows ---
all_labels_idx = df_full[label_col].map(class_to_idx).to_numpy(dtype=int)

# --- Draw the circle and anchor labels ---
theta_circle = np.linspace(0, 2 * np.pi, 300)
fig, ax = plt.subplots(figsize=(8, 8))
ax.plot(np.cos(theta_circle), np.sin(theta_circle), 'k-', lw=1)  # unit circle

# Mark anchor points and label them
for k in range(n_dims):
    ax.plot(anchor_x[k], anchor_y[k], 'k^', markersize=8)
    # Offset label slightly outward from the circle
    label_offset = 1.12
    ax.text(anchor_x[k] * label_offset, anchor_y[k] * label_offset,
            feature_cols_7[k], ha='center', va='center', fontsize=9)

# Plot each class
for i, name in enumerate(class_names):
    mask = (all_labels_idx == i)
    ax.scatter(proj_x[mask], proj_y[mask],
               s=10, alpha=0.6, label=name,
               color=COLORS[i % len(COLORS)])

ax.set_aspect('equal')
ax.set_title('Task 14 – Circle Segment Plot (first 7 dimensions)')
ax.legend(title='Class', loc='lower right', fontsize=8)
ax.axis('off')
plt.tight_layout()
plt.show()

## Task 15 – Parallel Coordinates Plot for first 7 dimensions (coloured by class)

In [ ]:
# -----------------------------------------------------------------------
# Parallel Coordinates Plot
# 7 equidistant vertical axes, one per dimension.
# Each data point is drawn as a polyline connecting its value on each axis.
# Line colour depends on the class label.
# Axes are labelled with feature names and show min/max tick values.
# -----------------------------------------------------------------------

# Build the dimension array (first 7 features) + numeric class labels
X7_full   = df_full[feature_cols_7].to_numpy(dtype=float)   # (n, 7)
labels_all = all_labels_idx                                   # (n,)

# --- Min/max normalise each axis to [0, 1] so all axes are on the same scale ---
col_min  = X7_full.min(axis=0)    # (7,)
col_max  = X7_full.max(axis=0)    # (7,)
col_range = col_max - col_min
col_range[col_range == 0] = 1     # guard against constant columns
X7_scaled = (X7_full - col_min) / col_range   # values in [0, 1], shape (n, 7)

# --- Set up figure ---
fig, ax = plt.subplots(figsize=(13, 6))

# X positions for the 7 equidistant axes
axis_x = np.arange(n_dims)   # [0, 1, 2, 3, 4, 5, 6]

# Draw a thin line for each data point
for row_idx in range(X7_scaled.shape[0]):
    cls  = labels_all[row_idx]
    # Use a low alpha so overlapping lines are still visible
    ax.plot(axis_x, X7_scaled[row_idx, :],
            color=COLORS[cls % len(COLORS)],
            alpha=0.15, lw=0.7)

# Draw solid vertical axis lines on top
for k in range(n_dims):
    ax.axvline(x=k, color='black', lw=1.5)

# Add axis labels (feature names) and min/max tick values
for k in range(n_dims):
    ax.text(k, -0.07, feature_cols_7[k], ha='center', va='top', fontsize=8, rotation=15)
    # Actual (un-normalised) min and max as tick labels
    ax.text(k, 0.0,  f'{col_min[k]:.1f}', ha='center', va='top',    fontsize=7, color='grey')
    ax.text(k, 1.0,  f'{col_max[k]:.1f}', ha='center', va='bottom', fontsize=7, color='grey')

# Add a legend patch for each class
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], color=COLORS[i % len(COLORS)], lw=2, label=name)
    for i, name in enumerate(class_names)
]
ax.legend(handles=legend_elements, title='Class', loc='upper right', fontsize=8)

ax.set_xlim(-0.3, n_dims - 0.7)
ax.set_ylim(-0.15, 1.1)
ax.set_xticks([])        # hide default x ticks; we drew custom labels above
ax.set_yticks([0, 0.5, 1])
ax.set_yticklabels(['Min', '0.5', 'Max'], fontsize=8)
ax.set_title('Task 15 – Parallel Coordinates (first 7 dimensions, coloured by class)')
plt.tight_layout()
plt.show()